# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset is described with a Croissant schema, located at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install the latest mlcroissant package
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and records using the `mlcroissant` API. 

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Set Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let us list available record sets (`@id`), fields and columns. We will reference all entities by their `@id`.

First, get available record set IDs and list their fields by `@id`.

In [ ]:
# Get all record set objects from metadata
record_sets = dataset.metadata.record_sets

print("Available record sets and fields:")
all_record_set_ids = []
for rs in record_sets:
    rs_id = rs.id
    all_record_set_ids.append(rs_id)
    print(f' - RecordSet @id: {rs_id}')
    print(f'   Fields:')
    for field in rs.fields:
        print(f'      · Field @id: {field.id} (dataType: {getattr(field, "data_type", "n/a")})')
    columns = getattr(rs, 'columns', [])
    if columns:
        print(f'   Columns:')
        for col in columns:
            print(f'      · Column @id: {col.id} (name: {getattr(col, "name", "n/a")}, dataType: {getattr(col, "data_type", "n/a")})')

## 3. Data Extraction

Now load all records from each record set using `mlcroissant`. For each, we load into a separate pandas DataFrame.

All entities are referenced by their full `@id` as shown above.

In [ ]:
# We'll extract all record sets found above
dataframes = {}
for rs_id in all_record_set_ids:
    # Fetch records as list of dicts for each record set
    try:
        rows = list(dataset.records(record_set=rs_id))
        if rows:
            df = pd.DataFrame(rows)
            dataframes[rs_id] = df
            print(f"\nLoaded DataFrame for RecordSet {rs_id}, shape: {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
        else:
            print(f"\nNo records loaded for RecordSet {rs_id}")
    except Exception as e:
        print(f"\nError loading records for {rs_id}: {e}")

# For further analysis let's use the first non-empty record set
main_rs_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_rs_id = k
        break

if main_rs_id:
    print(f"\nPreview of records from RecordSet {main_rs_id}:")
    display(dataframes[main_rs_id].head())
else:
    print('No records found in any RecordSet.')

## 4. Exploratory Data Analysis (EDA)

Let's perform some standard data wrangling operations. We'll:
- Select a numeric field by its `@id` (shown above)
- Filter records based on a value threshold
- Normalize the field
- Optionally, group by a categorical field (`@id`)

**Note:** If the field or RecordSet IDs are unknown, check the overview above.

In [ ]:
# Example: Choose a numeric field by its @id present in your DataFrame
# For demonstration, we'll try to find the first float/integer column

df = dataframes[main_rs_id] if main_rs_id else None
if df is not None:
    # Guess a numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field:
        print(f'Using numeric field: {numeric_field}')
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical (non-numeric) field, if available
        group_field = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() < len(df) // 2:
                group_field = col
                break

        if group_field:
            print(f"\nGrouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field detected in the current DataFrame.")
else:
    print('No DataFrame available for EDA.')

## 5. Visualization

Visualize distributions or trends for one or more fields.

- We will use matplotlib for standard plots.

In [ ]:
import matplotlib.pyplot as plt

if df is not None and numeric_field:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=12)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        df.groupby(group_field)[numeric_field].mean().plot(kind='bar')
        plt.title(f'Average {numeric_field} by {group_field}')
        plt.ylabel(f'Mean {numeric_field}')
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion

- Explored the FAIR^2 dataset using the Croissant specification via `mlcroissant`.
- Provided overview and step-by-step extraction of all record sets and fields by `@id`.
- Demonstrated typical data wrangling and filtering, emphasizing the correct use of entity `@ids`.
- Performed basic EDA and data visualization. 

**For deeper clinical or statistical analyses, consult the accompanying project documentation or CROISSANT schema.**